## MSPE-SwinUNETR full training (ABLATION DRIFT)



Vessels Dataset: 
https://figshare.com/s/0325c9e375930ef76795?file=62063935.

In [ ]:
!python -c "import monai" || pip install -q "monai[nibabel, tqdm]"
!python -c "import matplotlib" || pip install -q matplotlib
!python -c "import wandb" || pip install -q wandb
%matplotlib inline

In [ ]:
import os
import sys

colab = True
if colab:
    from google.colab import drive, runtime, userdata
    drive.mount('/content/drive')
    dataset_dir = r"/content/drive/MyDrive/DATASETS/HISTOPOLOGY/"
    root_dir = r"/content/drive/MyDrive/RUNS/HISTOPOLOGY/ABLATION/"
    mspe_code_dir = r"/content/drive/MyDrive/MSPE/"
    sys.path.append(mspe_code_dir)
    num_workers = 6 # Google G2 instance
    cache_rate = 1.0
else:
    dataset_dir = r""
    root_dir = r""
    mspe_code_dir = r""
    sys.path.append(mspe_code_dir)
    num_workers = 0
    cache_rate = 0.0
    userdata = None
    
os.makedirs(root_dir, exist_ok=True)

In [ ]:
import glob
import os
import sys
import time
import random
import copy
from abc import ABC, abstractmethod

import nibabel as nib
import numpy as np
import monai
import wandb
import tqdm
import matplotlib.pyplot as plt


from monai.data import CacheDataset, Dataset, DataLoader, list_data_collate

from monai.apps import CrossValidation
from monai.config import print_config
from monai.data import CacheDataset, Dataset, create_test_image_3d, list_data_collate, decollate_batch, DataLoader, PILReader, ThreadDataLoader
from monai.data.utils import partition_dataset
from monai.handlers import (
    MeanDice,
    MLFlowHandler,
    StatsHandler,
    TensorBoardImageHandler,
    TensorBoardStatsHandler,
)
from monai.inferers import sliding_window_inference
from monai.losses import TverskyLoss, DiceLoss, DiceCELoss, MaskedDiceLoss
from monai.metrics import DiceMetric, HausdorffDistanceMetric
from monai.networks.blocks import PatchEmbed
from monai.networks.layers import trunc_normal_, Conv, Norm
from monai.networks.nets import UNet
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, EnsureTyped,
    RandAffined, RandFlipd,RandRotated, RandScaleIntensityd, RandAdjustContrastd,
    DivisiblePadd, CropForegroundd,
    ScaleIntensityRanged, ScaleIntensityRangePercentilesd,
    AsDiscrete, AsDiscreted, CastToTyped, ToTensord,
)
from monai.utils import first, set_determinism

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler
from torch.optim.lr_scheduler import StepLR, LambdaLR, LinearLR, CosineAnnealingLR, SequentialLR

from swin_mspe import (
    MSPEPatchEmbedSwinNaiveRouting,
    MSPEPatchEmbedSwinOverlapping,
    MSPEPatchEmbedSwinDilatingK3,
    DEFAULT_RESOLUTIONS,
    DEFAULT_K,
    mspe_swin_train_step,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# bfloat16 on Ampere and later for autocast
AMP_DTYPE = torch.bfloat16 if (device.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
print(f"AMP dtype: {AMP_DTYPE}")
print_config()

In [ ]:
full_dataset = []
images_list = glob.glob(dataset_dir + "/raw_images/*.png")
# NOTE: Expert 1, vascular wall annotations
masks_list = glob.glob(dataset_dir + "/expert_1/vascular_wall/*.png")
roi_masks_list = glob.glob(dataset_dir + "/roi_masks/*.png")

# Sort images, masks for reproducibility
images_list.sort()
masks_list.sort()
roi_masks_list.sort()

assert len(images_list) == len(masks_list) == len(roi_masks_list), \
    f"Mismatch: {len(images_list)} images, {len(masks_list)} masks, {len(roi_masks_list)} ROI masks"

for img, mask, roi in tqdm.tqdm(zip(images_list, masks_list, roi_masks_list), total=len(images_list)):
    full_dataset.append({
        "image": img,
        "label": mask,
        "roi_mask": roi,
    })

print(f"Total samples: {len(full_dataset)}")

In [ ]:
# Set fold for 5-fold cross-validation

class CVDataset(Dataset):
    
    def __init__(self, data, cache_rate=cache_rate, num_workers=num_workers, transform=None):
        data = self._split_datalist(datalist=data)
        super().__init__(data=data, transform=transform)

    def _split_datalist(self, datalist):
        raise NotImplementedError(f"Subclass {self.__class__.__name__} must implement this method.")


FOLD_NUMBER = 0
total_folds = 5
folds = list(range(total_folds))

cv_splitter = CrossValidation(
    dataset_cls=CVDataset,
    data=full_dataset,
    nfolds=total_folds,
    seed=2026,
)

train_fold_ds = cv_splitter.get_dataset(
    folds=folds[:FOLD_NUMBER] + folds[FOLD_NUMBER + 1:],
    transform=None,
)
val_fold_ds = cv_splitter.get_dataset(
    folds=FOLD_NUMBER,
    transform=None,
)

train_dataset = list(train_fold_ds.data)
val_dataset = list(val_fold_ds.data)

print(f"Fold {FOLD_NUMBER}: Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
train_transforms = Compose([
    LoadImaged(keys=["image"]),
    LoadImaged(keys=["label", "roi_mask"], reader=PILReader(converter=lambda image: image.convert("L"))),
    AsDiscreted(keys=["label", "roi_mask"], threshold=127),
    EnsureChannelFirstd(keys=["image", "label", "roi_mask"]),
    ScaleIntensityRanged(keys=["image"], a_min=0.0, a_max=255.0, b_min=0.0, b_max=1.0, clip=True),
    EnsureTyped(keys=["image", "label", "roi_mask"]),
    CastToTyped(keys=["image", "label", "roi_mask"], dtype=(torch.float32, torch.uint8, torch.uint8)),
    
    # MSPE augmentation pipeline, with low probs 
    RandFlipd(keys=["image", "label", "roi_mask"], prob=0.2, spatial_axis=1), 
    RandFlipd(keys=["image", "label", "roi_mask"], prob=0.2, spatial_axis=0), 
    RandRotated(
        keys=["image", "label", "roi_mask"],
        prob=0.2,
        range_x=np.deg2rad(15),
        mode=("bilinear", "nearest", "nearest"),
        padding_mode="zeros",
    ),
    RandScaleIntensityd(keys=["image"], factors=0.10, prob=0.1),      
    RandAdjustContrastd(keys=["image"], gamma=(0.9, 1.1), prob=0.1),
    # For Swin
    DivisiblePadd(keys=["image", "label", "roi_mask"], k=32, mode="constant"),
])

val_transforms = Compose([
    LoadImaged(keys=["image"]),
    LoadImaged(keys=["label", "roi_mask"], reader=PILReader(converter=lambda image: image.convert("L"))),
    AsDiscreted(keys=["label", "roi_mask"], threshold=127),
    EnsureChannelFirstd(keys=["image", "label", "roi_mask"]),
    ScaleIntensityRanged(keys=["image"], a_min=0.0, a_max=255.0, b_min=0.0, b_max=1.0, clip=True),
    EnsureTyped(keys=["image", "label", "roi_mask"]),
    CastToTyped(keys=["image", "label", "roi_mask"], dtype=(torch.float32, torch.uint8, torch.uint8)),

    # For Swin
    DivisiblePadd(keys=["image", "label", "roi_mask"], k=32, mode="constant"),  
])

train_ds = CacheDataset(data=train_dataset, transform=train_transforms,
                        cache_rate=cache_rate, num_workers=num_workers)
train_loader = ThreadDataLoader(train_ds, num_workers=num_workers,
                                batch_size=1, shuffle=True,
                                pin_memory=True, persistent_workers=False,
                                collate_fn=list_data_collate)

val_ds = CacheDataset(data=val_dataset, transform=val_transforms,
                      cache_rate=cache_rate, num_workers=num_workers)
val_loader = ThreadDataLoader(val_ds, num_workers=num_workers,
                              batch_size=1, shuffle=False)

# Test is the same as val for cross-val
test_ds = val_ds
test_loader = val_loader

print(f"\nTrain batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
def display_sample_pairs(data_loader, n_samples=3):
    fig, axes = plt.subplots(n_samples, 4, figsize=(20, 4 * n_samples))
    if n_samples == 1:
        axes = [axes]

    for idx, batch in enumerate(data_loader):
        if idx >= n_samples:
            break

        image = batch["image"][0].cpu().permute(1, 2, 0)
        label = batch["label"][0][0].cpu()
        roi_mask = batch["roi_mask"][0][0].cpu()
        filename = os.path.basename(batch["image"].meta["filename_or_obj"][0])

        ax_row = axes[idx]
        ax_row[0].imshow(image)
        ax_row[0].set_title(f"Image\n{filename}", fontsize=9)
        ax_row[0].axis("off")
        ax_row[1].imshow(label, cmap="gray")
        ax_row[1].set_title(f"Mask (vascular wall)\n{filename}", fontsize=9)
        ax_row[1].axis("off")
        ax_row[2].imshow(roi_mask, cmap="gray")
        ax_row[2].set_title(f"ROI Mask\n{filename}", fontsize=9)
        ax_row[2].axis("off")
        ax_row[3].imshow(image, cmap="gray")
        ax_row[3].imshow(roi_mask, cmap="Blues", alpha=0.35)
        ax_row[3].imshow(label, cmap="Reds", alpha=0.45)
        ax_row[3].set_title(f"Mask + ROI Overlay\n{filename}", fontsize=9)
        ax_row[3].axis("off")

    plt.tight_layout()
    plt.show()

# Take a look at train data 
display_sample_pairs(train_loader, n_samples=3)

In [ ]:
# Experiment configuration

FEATURE_SIZE = 24
MAX_EPOCHS = 2
VAL_INTERVAL = 2
EARLY_STOPPING_PATIENCE = 40

LR = 1e-3
WD = 1e-5

WANDB_PROJECT = "VESSELS_MSPE_SWIN_ALL"
# train res are [256, 384, 512, 768, 1024, 1280]
TRAIN_RESOLUTIONS = list(DEFAULT_RESOLUTIONS)
# test res are [256, 384, 512, 768, 1024, 1280]
TEST_RESOLUTIONS = list(DEFAULT_RESOLUTIONS)
FOLD_TAG = f"FOLD_{FOLD_NUMBER}"

post_pred = Compose([AsDiscrete(argmax=True, to_onehot=2)])
post_label = Compose([AsDiscrete(to_onehot=2)])

# All train versions
EXPERIMENTS = [
    # {
    #     "name": "BASELINE_SWIN_V2",
    #     "description": "SwinUNETR V2 baseline, native-resolution training",
    #     "patch_embed_class": None,
    #     "use_v2": True,
    #     "train_mode": "native",
    # },
    # {
    #     "name": "SWIN_MIXED_V2",
    #     "description": "SwinUNETR V2, mixed-resolution training",  # For ablation 
    #     "patch_embed_class": None,
    #     "use_v2": True,
    #     "train_mode": "resize_aug",
    # },
    {
        "name": "ROUTING_SWIN",
        "description": "MSPE naive routing, full-model training",
        "patch_embed_class": MSPEPatchEmbedSwinNaiveRouting,
        "use_v2": True,
        "train_mode": "mspe",
    },
    {
        "name": "OVERLAPPING_SWIN",
        "description": "MSPE overlapping kernels, full-model training",
        "patch_embed_class": MSPEPatchEmbedSwinOverlapping,
        "use_v2": True,
        "train_mode": "mspe",
    },
    {
        "name": "DIALATING_SWIN",
        "description": "MSPE dilating kernels, full-model training",
        "patch_embed_class": MSPEPatchEmbedSwinDilatingK3,
        "use_v2": True,
        "train_mode": "mspe",
    },
]

# Fold-specific run names and checkpoints
for exp in EXPERIMENTS:
    exp["run_name"] = f"{exp['name']}_{FOLD_TAG}"
    exp["checkpoint_path"] = os.path.join(root_dir, f"best_metric_{exp['run_name']}.pth")

print(f"Experiments to run: {[e['run_name'] for e in EXPERIMENTS]}")
print(f"Train resolutions: {TRAIN_RESOLUTIONS}")
print(f"Test resolutions: {TEST_RESOLUTIONS}")
print(f"\nEpochs: {MAX_EPOCHS}, optimizer: AdamW, lr: {LR}, wd: {WD}")

In [ ]:
# Create defult Swin embedding for all baselines
def create_patch_embed(exp_config):
    patch_embed_class = exp_config["patch_embed_class"]
    if patch_embed_class is None:
        return PatchEmbed(patch_size=2, in_chans=3, embed_dim=FEATURE_SIZE, norm_layer=nn.LayerNorm, spatial_dims=2)

# For all MSPEs
    return patch_embed_class(
        patch_size=2,
        in_chans=3,
        embed_dim=FEATURE_SIZE,
        norm_layer=nn.LayerNorm,
        spatial_dims=2,
        K=DEFAULT_K,
        resolutions=DEFAULT_RESOLUTIONS,
    )

# Create SwinUntetr
def create_model_for_experiment(exp_config, device):
    model = monai.networks.nets.SwinUNETR(
        in_channels=3,
        out_channels=2,
        spatial_dims=2,
        feature_size=FEATURE_SIZE,
        use_v2=exp_config["use_v2"],
    ).to(device)

    model.swinViT.patch_embed = create_patch_embed(exp_config).to(device)
    print(f"\nCreated {exp_config['name']}")
    print(f"use_v2={exp_config['use_v2']}, train_mode={exp_config['train_mode']}")
    print(model.swinViT.patch_embed)
    return model

# Image AR-preserving resize 
def get_aspect_preserving_target_size(inputs, effective_resolution, divisor=32):
    spatial_shape = inputs.shape[2:]
    spatial_dims = len(spatial_shape)
    current_eff = float(np.prod(spatial_shape)) ** (1.0 / spatial_dims)
    scale = effective_resolution / current_eff

    target_size = []
    for dim in spatial_shape:
        resized_dim = int(round(dim * scale))
        resized_dim = max(divisor, int(round(resized_dim / divisor)) * divisor)
        target_size.append(resized_dim)
    return target_size

# Batch resize
def resize_batch(inputs, labels, roi_masks, effective_resolution):
    if effective_resolution is None:
        return inputs, labels, roi_masks

    spatial_dims = inputs.ndim - 2
    target_size = get_aspect_preserving_target_size(inputs, effective_resolution)
    image_mode = "bilinear" if spatial_dims == 2 else "trilinear"

    inputs = F.interpolate(inputs, size=target_size, mode=image_mode, align_corners=False)
    labels = F.interpolate(labels.float(), size=target_size, mode="nearest").to(labels.dtype)
    roi_masks = F.interpolate(roi_masks.float(), size=target_size, mode="nearest").to(roi_masks.dtype)
    return inputs, labels, roi_masks

# Train batch resize
def resize_train_batch(inputs, labels, roi_masks, effective_resolution=None):
    if effective_resolution is None:
        effective_resolution = int(np.random.choice(TRAIN_RESOLUTIONS))
    inputs, labels, roi_masks = resize_batch(inputs, labels, roi_masks, effective_resolution)
    return inputs, labels, roi_masks, int(effective_resolution), tuple(inputs.shape[2:])

# WB login
def login_wandb():
    if colab:
        wandb.login(key=userdata.get("WANDB_API_KEY"))

In [ ]:
# Print all metrics
def print_metric_table(title, dice_vals, hd95_vals):
    num_classes = hd95_vals.shape[0]
    print(title)
    print(f"{'Class':>8} | {'Dice':>8} | {'HD95':>10}")
    print("-" * 32)
    for c in range(num_classes):
        d = dice_vals[c].item()
        h = hd95_vals[c].item()
        print(f"{c+1:>8} | {d:>8.4f} | {h:>10.2f}")

    mean_dice = dice_vals.nanmean().item()
    mean_hd95 = hd95_vals.nanmean().item()
    print("-" * 32)
    print(f"{'Mean':>8} | {mean_dice:>8.4f} | {mean_hd95:>10.2f}")
    print()
    return mean_dice, mean_hd95

# Log metrics 
def log_eval_result(prefix, result, best_metric_epoch):
    if wandb.run is None:
        return

    log_data = {
        f"{prefix}/mean_dice": result["mean_dice"],
        f"{prefix}/mean_hd95": result["mean_hd95"],
        "best_metric_epoch": best_metric_epoch,
    }
    for c, (dice_value, hd95_value) in enumerate(zip(result["dice_vals"], result["hd95_vals"]), start=1):
        log_data[f"{prefix}/class_{c}_dice"] = dice_value.item()
        log_data[f"{prefix}/class_{c}_hd95"] = hd95_value.item()
    wandb.log(log_data)

# Compute evals helper 
def evaluate_test_loader(model, data_loader, resolution=None, prefix="test_native", best_metric_epoch=-1):
    hd95_per_class = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean_batch")
    dice_per_class = DiceMetric(include_background=False, reduction="mean_batch")

    with torch.no_grad():
        for test_data in data_loader:
            test_inputs, test_labels, test_roi_masks = (
                test_data["image"].to(device),
                test_data["label"].to(device),
                test_data["roi_mask"].to(device),
            )
            test_inputs, test_labels, test_roi_masks = resize_batch(
                test_inputs, test_labels, test_roi_masks, resolution
            )

            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                test_outputs = model(test_inputs)

            test_outputs_list = [post_pred(i) for i in decollate_batch(test_outputs)]
            test_labels_list = [post_label(i) for i in decollate_batch(test_labels)]
            test_roi_masks_list = decollate_batch(test_roi_masks)

            test_outputs_list = [pred * roi.to(dtype=pred.dtype) for pred, roi in zip(test_outputs_list, test_roi_masks_list)]
            test_labels_list = [label * roi.to(dtype=label.dtype) for label, roi in zip(test_labels_list, test_roi_masks_list)]

            hd95_per_class(y_pred=[v.cpu() for v in test_outputs_list], y=[v.cpu() for v in test_labels_list])
            dice_per_class(y_pred=[v.cpu() for v in test_outputs_list], y=[v.cpu() for v in test_labels_list])

    hd95_vals = hd95_per_class.aggregate()
    dice_vals = dice_per_class.aggregate()
    hd95_per_class.reset()
    dice_per_class.reset()

    title = "\nNative test metrics" if resolution is None else f"Resized test metrics at effective res {resolution}"
    mean_dice, mean_hd95 = print_metric_table(title, dice_vals, hd95_vals)
    result = {
        "dice_vals": dice_vals,
        "hd95_vals": hd95_vals,
        "mean_dice": mean_dice,
        "mean_hd95": mean_hd95,
    }
    log_eval_result(prefix, result, best_metric_epoch)
    return result

# Compute evals for all resolutions 
def evaluate_all_resolutions(model, test_loader, exp_name, best_metric_epoch, checkpoint_path):
    print(f"\n{'=' * 40}")
    print(f"TEST EVALUATION: {exp_name}")
    print(f"{'=' * 40}")

    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    model.eval()

    native_result = evaluate_test_loader(
        model, test_loader, resolution=None,
        prefix="test_native", best_metric_epoch=best_metric_epoch,
    )

    multi_resolution_results = {}
    for hw in TEST_RESOLUTIONS:
        multi_resolution_results[hw] = evaluate_test_loader(
            model, test_loader, resolution=hw,
            prefix=f"test_res_{hw}", best_metric_epoch=best_metric_epoch,
        )

    if wandb.run is not None:
        wandb.log({
            "test/mean_dice": native_result["mean_dice"],
            "test/mean_hd95": native_result["mean_hd95"],
            "best_metric_epoch": best_metric_epoch,
        })

    return {"native": native_result, "multi_res": multi_resolution_results}

In [ ]:
# See kernels drift
def analyze_kernel_drift(initial_state, model, exp_name):
    patch_embed = model.swinViT.patch_embed
    if not hasattr(patch_embed, "patch_kernels"):
        print(f"{exp_name}: no MSPE kernels; skipping drift analysis.")
        return None

    final_state = patch_embed.state_dict()
    metrics_l2 = []
    metrics_cos = []
    metrics_max = []

    print(f"\nKernel drift analysis: {exp_name}")
    print("Kernel | L2 Dist  | Cosine Sim | Max Diff")
    print("-" * 44)

    for k in range(len(patch_embed.patch_kernels)):
        weight_key = f"patch_kernels.{k}.weight"
        if weight_key not in initial_state or weight_key not in final_state:
            raise KeyError(f"Missing MSPE weight key for drift analysis: {weight_key}")

        w_init = initial_state[weight_key].detach().flatten().cpu()
        w_final = final_state[weight_key].detach().flatten().cpu()
        l2_dist = torch.norm(w_final - w_init, p=2).item()
        cos_sim = F.cosine_similarity(w_final.unsqueeze(0), w_init.unsqueeze(0)).item()
        max_diff = torch.max(torch.abs(w_final - w_init)).item()

        metrics_l2.append(l2_dist)
        metrics_cos.append(cos_sim)
        metrics_max.append(max_diff)
        print(f"{k:>6} | {l2_dist:>8.4f} | {cos_sim:>10.4f} | {max_diff:>8.4f}")

    drift = {
        "l2": metrics_l2,
        "cosine": metrics_cos,
        "max_diff": metrics_max,
    }

    if wandb.run is not None:
        log_data = {}
        for k, (l2_dist, cos_sim, max_diff) in enumerate(zip(metrics_l2, metrics_cos, metrics_max)):
            log_data[f"kernel_drift/kernel_{k}_l2"] = l2_dist
            log_data[f"kernel_drift/kernel_{k}_cosine"] = cos_sim
            log_data[f"kernel_drift/kernel_{k}_max_diff"] = max_diff
        wandb.log(log_data)

    x = np.arange(len(metrics_l2))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].bar(x, metrics_l2)
    axes[0].set_title("L2 distance")
    axes[0].set_xlabel("Kernel")
    axes[1].bar(x, metrics_cos)
    axes[1].set_title("Cosine similarity")
    axes[1].set_xlabel("Kernel")
    axes[2].bar(x, metrics_max)
    axes[2].set_title("Max absolute diff")
    axes[2].set_xlabel("Kernel")
    fig.suptitle(f"MSPE kernel drift: {exp_name}")
    plt.tight_layout()

    if wandb.run is not None:
        wandb.log({"kernel_drift/plot": wandb.Image(fig)})
    plt.show()

    return drift

In [ ]:
# Per-epoch MSPE kernel-drift logging helper (cosine-to-init)
def kernel_cosines_to_init(patch_embed, initial_state):
    """Per-kernel cosine(weight_now, weight_init); returns list[float] of length K."""

    cur = patch_embed.state_dict()
    cos = []
    
    for k in range(len(patch_embed.patch_kernels)):
        key = f"patch_kernels.{k}.weight"
        w0 = initial_state[key].detach().flatten().float().cpu()
        w = cur[key].detach().flatten().float().cpu()
        cos.append(F.cosine_similarity(w.unsqueeze(0), w0.unsqueeze(0)).item())
    return cos

In [ ]:
# Train helpers

# Move to device and resize batch
def _prepare_training_batch(batch_data, exp_config):
    inputs, labels, roi_masks = (
        batch_data["image"].to(device),
        batch_data["label"].to(device),
        batch_data["roi_mask"].to(device),
    )

    if exp_config["train_mode"] == "resize_aug":
        inputs, labels, roi_masks, _, _ = resize_train_batch(inputs, labels, roi_masks)

    return inputs, labels, roi_masks

# Compute loss
def _compute_training_loss(model, inputs, labels, roi_masks, loss_function, exp_config):
    if exp_config["train_mode"] == "mspe":
        # NOTE: MSPE train step performs multi-res routing
        return mspe_swin_train_step(model, inputs, labels, loss_function, lam=1.0, mask=roi_masks)

    outputs = model(inputs)
    return loss_function(outputs, labels, mask=roi_masks)

# Training loop
def train_variant(model, train_loader, val_loader, exp_config, initial_mspe_state=None):
    exp_name = exp_config["name"]
    checkpoint_path = exp_config["checkpoint_path"]

    # loss_function 
    masked_dice = MaskedDiceLoss(include_background=False, to_onehot_y=True, softmax=False)
    loss_function = lambda logits, target, mask=None: masked_dice(logits.softmax(dim=1), target, mask=mask)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    scheduler = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)
    dice_metric = DiceMetric(include_background=False, reduction="mean")
    scaler = GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))

    best_metric = -1
    best_metric_epoch = -1
    epoch_loss_values = []
    metric_values = []
    epochs_no_improve = 0
    completed_epochs = 0
    total_start = time.time()

    # Per-epoch MSPE kernel-drift logging (cosine-to-init). None for baselines.
    patch_embed = model.swinViT.patch_embed
    if hasattr(patch_embed, "patch_kernels"):
        drift_init_state = initial_mspe_state or copy.deepcopy(patch_embed.state_dict())
        drift_traj = [(0, kernel_cosines_to_init(patch_embed, drift_init_state))]  # epoch 0 ~= 1.0
    else:
        drift_init_state = None
        drift_traj = None

    for epoch in range(MAX_EPOCHS):
        epoch_start = time.time()
        print("-" * 10)
        print(f"{exp_name}: epoch {epoch + 1}/{MAX_EPOCHS}")
        model.train()
        epoch_loss = 0
        step = 0

        for batch_data in train_loader:
            step += 1
            inputs, labels, roi_masks = _prepare_training_batch(batch_data, exp_config)

            optimizer.zero_grad()
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                loss = _compute_training_loss(model, inputs, labels, roi_masks, loss_function, exp_config)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item()

            log_data = {"train/step_loss": loss.item(), "train/amp_scale": scaler.get_scale()}
            if wandb.run is not None:
                wandb.log(log_data)

            print(f"{step}/{len(train_loader)}, train_loss: {loss.item():.4f}")

        epoch_loss /= step
        epoch_loss_values.append(epoch_loss)
        completed_epochs = epoch + 1
        epoch_time_sec = time.time() - epoch_start
        current_lr = optimizer.param_groups[0]["lr"]

        print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")
        print(f"time per epoch: {epoch_time_sec:.2f} s")

        if wandb.run is not None:
            wandb.log({
                "train/epoch_loss": epoch_loss,
                "lr": current_lr,
                "epoch": epoch + 1,
                "train/time_per_epoch": epoch_time_sec,
            })

        # Record per-kernel cosine-to-init for this epoch (MSPE variants only)
        if drift_traj is not None:
            epoch_cos = kernel_cosines_to_init(patch_embed, drift_init_state)
            drift_traj.append((epoch + 1, epoch_cos))
            if wandb.run is not None:
                cos_log = {f"kernel_drift/kernel_{k}_cosine": c for k, c in enumerate(epoch_cos)}
                cos_log["epoch"] = epoch + 1
                wandb.log(cos_log)

        if (epoch + 1) % VAL_INTERVAL == 0:
            model.eval()
            with torch.no_grad():
                for val_data in val_loader:
                    val_inputs, val_labels, val_roi_masks = (
                        val_data["image"].to(device),
                        val_data["label"].to(device),
                        val_data["roi_mask"].to(device),
                    )
                    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                        val_outputs = model(val_inputs)

                    val_outputs_list = [post_pred(i) for i in decollate_batch(val_outputs)]
                    val_labels_list = [post_label(i) for i in decollate_batch(val_labels)]
                    val_roi_masks_list = decollate_batch(val_roi_masks)

                    val_outputs_list = [pred * roi.to(dtype=pred.dtype) for pred, roi in zip(val_outputs_list, val_roi_masks_list)]
                    val_labels_list = [label * roi.to(dtype=label.dtype) for label, roi in zip(val_labels_list, val_roi_masks_list)]
                    dice_metric(y_pred=val_outputs_list, y=val_labels_list)

                metric = dice_metric.aggregate().item()
                dice_metric.reset()
                metric_values.append(metric)

                if metric > best_metric:
                    best_metric = metric
                    best_metric_epoch = epoch + 1
                    epochs_no_improve = 0
                    torch.save(model.state_dict(), checkpoint_path)
                    print(f"saved new best metric model to {checkpoint_path}")
                    print(
                        f"current epoch: {epoch + 1} current mean dice: {metric:.3f}\n"
                        f"best validation mean dice: {best_metric:.3f} at epoch: {best_metric_epoch}"
                    )
                else:
                    epochs_no_improve += VAL_INTERVAL

                if wandb.run is not None:
                    wandb.log({"val Dice": metric, "epoch": epoch + 1})

                if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
                    print(f"Early stopping triggered at epoch {epoch + 1}. No improvement for {EARLY_STOPPING_PATIENCE} epochs.")
                    if wandb.run is not None:
                        wandb.log({"early_stop_epoch": epoch + 1})
                    break

        scheduler.step()

    total_time = time.time() - total_start
    avg_epoch_time = total_time / max(completed_epochs, 1)
    print(f"total epochs: {completed_epochs}, total training time: {total_time / 60:.2f} min, average time per epoch: {avg_epoch_time:.2f} s")
    print(f"train completed, best validation metric: {best_metric:.4f} at epoch: {best_metric_epoch}")

    return best_metric, best_metric_epoch, epoch_loss_values, metric_values, drift_traj

In [ ]:
# Experimenal loop

login_wandb()
all_results = {}
all_training_curves = {}
all_drift = {}
all_drift_traj = {}

for exp_idx, exp in enumerate(EXPERIMENTS):
    set_determinism(seed=2026)

    exp_name = exp["name"]
    run_name = exp["run_name"]
    print(f"\n{'=' * 60}")
    print(f"EXPERIMENT {exp_idx + 1}/{len(EXPERIMENTS)}: {run_name}")
    print(exp["description"])
    print(f"{'=' * 60}\n")

    wandb.init(
        project=WANDB_PROJECT,
        name=run_name,
        group=exp_name,
        tags=[FOLD_TAG],
        config={
            "max_epochs": MAX_EPOCHS,
            "val_interval": VAL_INTERVAL,
            "feature_size": FEATURE_SIZE,
            "gradient_checkpointing": False,
            "learning_rate": LR,
            "weight_decay": WD,
            "optimizer": "AdamW",
            "loss_type": "MaskedDiceLoss",
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "fold_number": FOLD_NUMBER,
            "total_fold_number": total_folds,
            "fold_tag": FOLD_TAG,
            "train_mode": exp["train_mode"],
            "use_v2": exp["use_v2"],
            "K": DEFAULT_K if exp["patch_embed_class"] else 1,
            "train_resolutions": TRAIN_RESOLUTIONS if exp["train_mode"] in ("resize_aug", "mspe") else None,
            "test_resolutions": TEST_RESOLUTIONS,
        },
    )

    model = create_model_for_experiment(exp, device)
    initial_mspe_state = None
    if exp["patch_embed_class"] is not None:
        initial_mspe_state = copy.deepcopy(model.swinViT.patch_embed.state_dict())

    best_metric, best_metric_epoch, epoch_losses, metric_vals, drift_traj = train_variant(
        model, train_loader, val_loader, exp, initial_mspe_state=initial_mspe_state
    )
    all_training_curves[exp_name] = {
        "epoch_losses": epoch_losses,
        "metric_values": metric_vals,
        "best_metric": best_metric,
        "best_metric_epoch": best_metric_epoch,
        "checkpoint_path": exp["checkpoint_path"],
    }
    if drift_traj is not None:
        all_drift_traj[exp_name] = drift_traj

    results = evaluate_all_resolutions(model, test_loader, exp_name, best_metric_epoch, checkpoint_path=exp["checkpoint_path"])
    all_results[exp_name] = results

    if initial_mspe_state is not None:
        all_drift[exp_name] = analyze_kernel_drift(initial_mspe_state, model, exp_name)

    if wandb.run is not None:
        wandb.finish()
        
    del model
    torch.cuda.empty_cache()
    print(f"\n--- Completed {exp_name}")

print("\n" + "=" * 60)
print("ALL FULL-TRAINING EXPERIMENTS COMPLETED")
print("=" * 60)

In [ ]:
# Summary tables

print(f"\n{'=' * 80}")
print("RESULTS SUMMARY: Full training")
print(f"{'=' * 80}\n")

res_cols = ["Native"] + [str(r) for r in TEST_RESOLUTIONS]
header = f"{'Condition':<26} | " + " | ".join(f"{c:>8}" for c in res_cols) + " |"
sep = "-" * len(header)

print("\nDice (class 1):")
print(sep)
print(header)
print(sep)
for exp_name, results in all_results.items():
    native_dice = results["native"]["mean_dice"]
    row = f"{exp_name:<26} | {native_dice:>8.4f}"
    for hw in TEST_RESOLUTIONS:
        dice = results["multi_res"][hw]["mean_dice"]
        row += f" | {dice:>8.4f}"
    row += " |"
    print(row)
print(sep)

print("\nHD95 (class 1):")
print(sep)
print(header)
print(sep)
for exp_name, results in all_results.items():
    native_hd95 = results["native"]["mean_hd95"]
    row = f"{exp_name:<26} | {native_hd95:>8.2f}"
    for hw in TEST_RESOLUTIONS:
        hd95 = results["multi_res"][hw]["mean_hd95"]
        row += f" | {hd95:>8.2f}"
    row += " |"
    print(row)
print(sep)

print("\nTraining Summary:")
print(f"{'Condition':<26} | {'Best Dice':>10} | {'Best Epoch':>10} | {'Final Loss':>10}")
print("-" * 68)
for exp_name, curves in all_training_curves.items():
    final_loss = curves["epoch_losses"][-1] if curves["epoch_losses"] else float("nan")
    print(f"{exp_name:<26} | {curves['best_metric']:>10.4f} | {curves['best_metric_epoch']:>10d} | {final_loss:>10.4f}")

if all_drift:
    print("\nMSPE Kernel Drift Summary:")
    drift_header = f"{'Condition':<26} | {'Kernel':>6} | {'L2':>8} | {'Cosine':>10} | {'Max Diff':>8}"
    print(drift_header)
    print("-" * len(drift_header))
    for exp_name, drift in all_drift.items():
        for k, (l2_dist, cos_sim, max_diff) in enumerate(zip(drift["l2"], drift["cosine"], drift["max_diff"])):
            print(f"{exp_name:<26} | {k:>6d} | {l2_dist:>8.4f} | {cos_sim:>10.4f} | {max_diff:>8.4f}")

In [ ]:
# Assemble per epoch kernel drift CSV for figures

import pandas as pd

DRIFT_COLS = {
    "OVERLAPPING_SWIN": ["k1", "k2", "k3"],  
    "DIALATING_SWIN":   ["d1", "d2", "d3"],
    "ROUTING_SWIN":     ["r1", "r2", "r3"],
}

frames = []
for exp_name, cols in DRIFT_COLS.items():
    traj = all_drift_traj.get(exp_name)
    if not traj:
        print(f"[drift] no trajectory for {exp_name}; skipping (was it trained this run?)")
        continue
    df = pd.DataFrame({
        "epoch": [e for e, _ in traj],
        **{c: [v[i] for _, v in traj] for i, c in enumerate(cols)},
    }).set_index("epoch")
    frames.append(df)

if frames:
    drift_df = pd.concat(frames, axis=1).reset_index().sort_values("epoch")
    drift_csv_path = os.path.join(root_dir, f"kernel_drift_{FOLD_TAG}.csv")
    drift_df.to_csv(drift_csv_path, index=False)
    print(f"[drift] saved {drift_csv_path}")
    print(f"[drift] {len(drift_df)} epochs x {drift_df.shape[1] - 1} kernels")
    print("Next: copy this file to the repo as figures/kernel_drift.csv, then run:")
    print("    python figures/plot_kernel_drift.py")
    display(drift_df.head())
else:
    print("[drift] no MSPE drift trajectories collected this run.")

In [ ]:
# Training curves

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
for exp_name, curves in all_training_curves.items():
    epochs = list(range(1, len(curves["epoch_losses"]) + 1))
    ax.plot(epochs, curves["epoch_losses"], label=exp_name)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training Loss")
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
for exp_name, curves in all_training_curves.items():
    val_epochs = [VAL_INTERVAL * (i + 1) for i in range(len(curves["metric_values"]))]
    ax.plot(val_epochs, curves["metric_values"], label=exp_name, marker="o", markersize=3)
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation Dice")
ax.set_title("Validation Dice")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Quick sanity check for trained checkpoints

def to_display_image(tensor_chw):
    image = tensor_chw.detach().cpu()
    if image.shape[0] in (3, 4):
        return image.permute(1, 2, 0)
    return image[0]


vis_resolutions = [None] + list(TEST_RESOLUTIONS)

login_wandb() 

for exp in EXPERIMENTS:
    ckpt = exp["checkpoint_path"]
    print(f"Visualizing {exp['name']} from {ckpt}")

    model_viz = create_model_for_experiment(exp, device)
    model_viz.load_state_dict(torch.load(ckpt, weights_only=True))
    model_viz.eval()

    with torch.no_grad():
        val_data = first(val_loader)
        val_inputs = val_data["image"].to(device)
        val_labels = val_data["label"].to(device)
        val_roi_masks = val_data["roi_mask"].to(device)

        n_cols = len(vis_resolutions)
        fig, axes = plt.subplots(4, n_cols, figsize=(5 * n_cols, 16))
        if n_cols == 1:
            axes = axes.reshape(4, 1)

        for col, effective_resolution in enumerate(vis_resolutions):
            inputs_r, labels_r, roi_r = resize_batch(
                val_inputs, val_labels, val_roi_masks, effective_resolution
            )

            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                outputs = model_viz(inputs_r)

            image = to_display_image(inputs_r[0])
            label = labels_r[0, 0].detach().cpu()
            roi_mask = roi_r[0, 0].detach().cpu()
            pred = torch.argmax(outputs, dim=1).detach().cpu()[0]
            pred = pred * roi_mask.to(dtype=pred.dtype)

            h, w = inputs_r.shape[-2], inputs_r.shape[-1]
            col_title = f"native\n{h}x{w}" if effective_resolution is None else f"eff {effective_resolution}\n{h}x{w}"

            axes[0, col].imshow(image, cmap="gray")
            axes[0, col].set_title(col_title)
            axes[0, col].axis("off")
            axes[1, col].imshow(label, cmap="viridis")
            axes[1, col].set_title("label")
            axes[1, col].axis("off")
            axes[2, col].imshow(pred, cmap="viridis")
            axes[2, col].set_title("prediction")
            axes[2, col].axis("off")
            axes[3, col].imshow(roi_mask, cmap="viridis")
            axes[3, col].set_title("ROI")
            axes[3, col].axis("off")

        fig.suptitle(exp["name"], fontsize=14)
        plt.tight_layout()

        viz_path = os.path.join(root_dir, f"viz_{exp['run_name']}.png")
        fig.savefig(viz_path, dpi=300, bbox_inches="tight")
        print(f"saved visualization to {viz_path}")

        # Save inference image to wandb
        wandb.init(
            project=WANDB_PROJECT,
            name=f"{exp['run_name']}_viz",
            group=exp["name"],
            tags=[FOLD_TAG, "viz"],
        )
        wandb.log({"predictions": wandb.Image(fig)})
        wandb.finish()
        
        plt.show()

    del model_viz
    torch.cuda.empty_cache()

In [ ]:
# Save requirements
# !pip freeze > /content/drive/MyDrive/MSPE/requirements_frozen.txt

In [ ]:
# Disconnect from runtime 
runtime.unassign()